# Análise Descritiva - Capítulo 44

Este notebook contém uma análise descritiva sobre os dados de itens de nota fiscal do Capítulo 44 do NCM, conforme dados disponibilizados em ```tb_treina_cap44.csv```

## 1. Configuração Inicial e Carregamento de Dados (Pandas)

In [ ]:
import pandas as pd
import re
import os
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from IPython.display import display
from gensim.models import Word2Vec, FastText

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

FILE_PATH = 'base_outros_a_analisar.csv'

# O arquivo é separado por tabulação ('\t')
try:
    df_pandas = pd.read_csv(FILE_PATH, sep=';', decimal='.')
    print(f"Total de linhas: {len(df_pandas)}")
    
    df_pandas['prod_vuntrib'] = pd.to_numeric(df_pandas['prod_vuntrib'], errors='coerce')
    df_pandas['prod_qtrib'] = pd.to_numeric(df_pandas['prod_qtrib'], errors='coerce')
    df_pandas['contagem'] = pd.to_numeric(df_pandas['contagem'], errors='coerce', downcast='integer')
    
    df_pandas.info()
    print("\nPrimeiras 5 linhas:")
    print(df_pandas.head())
except Exception as e:
    print(f"Ocorreu um erro durante o carregamento do Pandas: {e}")

In [ ]:
# Normalização para MAIÚSCULAS
df_pandas['prod_xprod_norm'] = df_pandas['prod_xprod'].astype(str).str.upper()
print(df_pandas['prod_xprod_norm'].head())

## 2. Análise de Contagem (`contagem`)

In [ ]:
# Ordenar os produtos por contagem (qtd) (tanto asc como desc)
print("Top 50 Produtos por Contagem (Mais Frequentes)")
top_50_contagem = df_pandas.sort_values(by='contagem', ascending=False).head(50)
print(top_50_contagem[['prod_xprod', 'contagem', 'prod_vuntrib']].reset_index(drop=True))

In [ ]:
print("\nBottom 50 Produtos por Contagem (Menos Frequentes)")
bottom_50_contagem = df_pandas.sort_values(by='contagem', ascending=True).head(50)
print(bottom_50_contagem[['prod_xprod', 'contagem', 'prod_vuntrib']].reset_index(drop=True))

## 3. Análise de Preço (`prod_vuntrib` e `prod_qtrib`)

In [ ]:
print("Top 50 Produtos por Valor Unitário Tributável (prod_vuntrib)")
top_50_vuntrib = df_pandas.sort_values(by='prod_vuntrib', ascending=False).head(50)
print(top_50_vuntrib[['prod_xprod', 'prod_vuntrib', 'contagem']].reset_index(drop=True))

In [ ]:
print("Bottom 50 Produtos por Valor Unitário Tributável (prod_vuntrib)")
bottom_50_vuntrib = df_pandas.sort_values(by='prod_vuntrib', ascending=True).head(50)
print(bottom_50_vuntrib[['prod_xprod', 'prod_vuntrib', 'contagem']].reset_index(drop=True))

In [ ]:
print("Top 50 Produtos por Quantidade Tributável (prod_qtrib)")
top_50_qtrib = df_pandas.sort_values(by='prod_qtrib', ascending=False).head(50)
print(top_50_qtrib[['prod_xprod', 'prod_qtrib', 'contagem']].reset_index(drop=True))

In [ ]:
print("Bottom 50 Produtos por Quantidade Tributável (prod_qtrib)")
bottom_50_qtrib = df_pandas.sort_values(by='prod_qtrib', ascending=True).head(50)
print(bottom_50_qtrib[['prod_xprod', 'prod_qtrib', 'contagem']].reset_index(drop=True))

## 4. Análise de Representatividade (Valor Total)

In [ ]:
# Criar colunas de representatividade (Valor Total = Valor Unitário * Contagem)
df_pandas['valor_total_vuntrib'] = df_pandas['prod_vuntrib'] * df_pandas['contagem']
df_pandas['valor_total_qtrib'] = df_pandas['prod_qtrib'] * df_pandas['contagem']

In [ ]:
print("Top 50 Produtos por Valor Total Tributável (valor_total_vuntrib)")
top_50_total_vuntrib = df_pandas.sort_values(by='valor_total_vuntrib', ascending=False).head(50)
print(top_50_total_vuntrib[['prod_xprod', 'prod_vuntrib', 'contagem', 'valor_total_vuntrib']].reset_index(drop=True))

In [ ]:
print("Bottom 50 Produtos por Valor Total Tributável (valor_total_vuntrib)")
bottom_50_total_vuntrib = df_pandas.sort_values(by='valor_total_vuntrib', ascending=True).head(50)
print(bottom_50_total_vuntrib[['prod_xprod', 'prod_vuntrib', 'contagem', 'valor_total_vuntrib']].reset_index(drop=True))

In [ ]:
print("Top 50 Produtos por Quantidade Total Tributável (valor_total_qtrib)")
top_50_total_qtrib = df_pandas.sort_values(by='valor_total_qtrib', ascending=False).head(50)
print(top_50_total_qtrib[['prod_xprod', 'prod_qtrib', 'contagem', 'valor_total_qtrib']].reset_index(drop=True))

In [ ]:
print("Bottom 50 Produtos por Quantidade Total Tributável (valor_total_qtrib)")
bottom_50_total_qtrib = df_pandas.sort_values(by='valor_total_qtrib', ascending=True).head(50)
print(bottom_50_total_qtrib[['prod_xprod', 'prod_qtrib', 'contagem', 'valor_total_qtrib']].reset_index(drop=True))

## Regex

In [ ]:
import re
import pandas as pd

# ==============================================================================
# 1. DICIONÁRIOS DE TERMOS E EXCLUSÕES (SUA BASE VALIDADA)
# ==============================================================================
EXCL_MATERIAIS = 'ALUMINIO|FERRO|METAL|INOX|PVC|VIDRO|PLASTICO|ACRILICO|CERAMICA|PORCELANA|CONCRETO|CIMENTO|GRANITO|MARMORE|PEDRA'
EXCL_MOVEIS = 'ARMARIO|ESTANTE|MESA|CADEIRA|SOFA|POLTRONA|RACK|CRIADO MUDO|CAMA|BERCO|GUARDA ROUPA|COMODA|APARADOR|BUFFET|CRISTALEIRA|COZINHA|PLANEJADA|PLANEJADOS'
EXCL_DECORACAO = 'BRINQUEDO|BONECA|MINIATURA|ARTESANATO|ENFEITE|JOIA|BIJUTERIA|APLIQUE|RECORTE|LASER'
EXCL_SERVICOS = 'LOCACAO|ALUGUEL|REFORMA|CONSERTO|MANUTENCAO|REPARO|FRETE|MONTAGEM|INSTALACAO|TAXA|ENTREGA|AJUSTE'
TERMOS_ESPECIES = 'CAMBARA|ANGELIM|TAUARI|CEDRINHO|CURUPIXA|ITAUBA|TATAJUBA|CUMARU|TECA|ABIURANA|CUPIUBA|AMESCLA|PINUS|EUCALIPTO|ARAUCO|GREENPLAC|CEDRO|MOGNO|VIROLA'

# ==============================================================================
# 2. ESTRUTURA DE CATEGORIAS (HIERARQUIA FISCAL NCM)
# ==============================================================================
categorias_regex = [
    # A. BLOQUEIO DE OUTLIERS
    ('OUTLIER__FINANCEIRO', rf'.*({EXCL_SERVICOS}|LOTE MISTO|MERCADORIA MISTA).*'),
    ('OUTLIER__MAQUINAS', r'(PICADOR|TRATOR|COLHEITADEIRA|MÁQUINA|MAQUINA|VEÍCULO|VEICULO)'),
    
    # B. ROTULADOS ANTES (Sua regra de ouro)
    ('ROTULADOS ANTES', r'(LENHA|TORA DE MADEIRA|MADEIRA CERRADA|CABO DE MADEIRA|ARTIGO DOMESTICO DE MADEIRA|CARVAO VEGETAL|PORTA-BATENTE DE MADEIRA|PALITO DE DENTE|PALLET-EMBALAGEM DE MADEIRA|CHAPAS DE MDF|CHAPAS DE COMPENSADO|CHAPAS DE TAPUME|RESIDUO DE MADEIRA)'),

    # C. NCM 44.01 a 44.06 (Matéria-Prima e Resíduos)
    ('44.01.LENHA_RESIDUOS', r'^(?=.*(LENHA|TORA|ACHA|ESTILHA|SERRAGEM|RESIDUO|CAVACO|BIOMASSA))(?!.*(ALUMINIO|FERRO|METAL|PVC)).*'),
    ('44.02.CARVAO_VEGETAL', r'^(?=.*(CARVAO|CARVÃO|MOINHA))(?!.*(ATIVADO|ATIVO|MINERAL|ANTRACIT|COQUE|HULHA|LINHITA|TURFA|FILTRO|MASCARA|SHAMPOO|SABONETE)).*'),
    ('44.03.MADEIRA_BRUTA', r'^(?=.*(MADEIRA.*(BRUTA|EM BRUTO)|TORA|TORO|RONCO))(?!.*(SERRADA|APARELHADA|BENEFICIADA)).*'),
    ('44.04.ARCOS_ESTACAS', r'^(?=.*(ARCO|ESTACA|MOIRAO|POSTE))(?=.*(MADEIRA)).*'),
    ('44.05.LA_FARINHA', r'^(?=.*(LA DE MADEIRA|FARINHA DE MADEIRA|FIBRA DE MADEIRA))(?!.*(MINERAL|VIDRO|SINTETICA)).*'),
    ('44.06.DORMENTES', r'^(?=.*(DORMENTE))(?=.*(MADEIRA|FERROVIA|TRILHO))(?!.*(CONCRETO|METAL)).*'),

    # D. NCM 44.07 a 44.09 (Madeira Processada)
    ('44.07.MADEIRA_SERRADA', rf'^(?=.*(TABUA|PRANCHA|VIGA|VIGOTA|CAIBRO|RIPA|SARRAFO|BARROTE|PONTALETE|{TERMOS_ESPECIES}))(?!.*(COMPENSADO|MDF|PAINEL)).*'),
    ('44.08.FOLHAS_FOLHEADOS', r'^(?=.*(FOLHA PARA FOLHEADO|FOLHEADO|LAMINA|DESENROLADA))(?=.*(6MM|6 MM|FINA|DELGADA)).*'),
    ('44.09.PERFILADOS', r'^(?=.*(MOLDURA|RODAPE|TACO|LAMBRI|RODAMEIO|ALIZAR))(?=.*(MADEIRA|PINUS|EUCALIPTO)).*'),

    # E. NCM 44.10 a 44.12 (Painéis e Chapas - GRANULARIDADE RECUPERADA)
    ('44.10.PAINEIS_PARTICULAS_OSB', rf'^(?!.*({EXCL_MOVEIS}))(?=.*(OSB|MDP|PARTICULA|AGLOMERADO))(?=.*(MADEIRA|MAD)).*'),
    ('44.11.PAINEIS_FIBRAS_MDF', rf'^(?!.*({EXCL_MOVEIS}))(?=.*(MDF|HDF|FIBRA|ARAUCO|GREENPLAC))(?=.*(18MM|15MM|3MM|MADEIRA|MAD)).*'),
    ('44.12.COMPENSADO', rf'^(?!.*({EXCL_MOVEIS}))(?=.*(COMPENSADO|CONTRAPLACADO|LAMINADO))(?=.*(MADEIRA|MAD|PINUS|EUCALIPTO)).*'),

    # F. NCM 44.13 a 44.17 (Obras Diversas)
    ('44.13.MADEIRA_DENSIFICADA', r'^(?=.*(DENSIFICADA|ESTABILIZADA))(?=.*(MADEIRA)).*'),
    ('44.14.MOLDURAS_QUADROS', rf'^(?=.*(MOLDURA))(?=.*(QUADRO|ESPELHO|FOTO))(?=.*(MADEIRA)).*'),
    ('44.15.CAIXOTES_PALETES', rf'^(?=.*(PALETE|PALLET|ESTRADO|CAIXOTE|ENGRADADO))(?!.*({EXCL_SERVICOS})).*'),
    ('44.16.BARRIS_TANOEIRO', r'^(?=.*(BARRIL|TONEL|ADUELA|CUBA|DORNA))(?=.*(MADEIRA)).*'),
    ('44.17.FERRAMENTAS_CABOS', r'^(?=.*(FERRAMENTA|CABO|PÁ|ENXADA|VASSOURA))(?=.*(MADEIRA)).*'),

    # G. NCM 44.18 (Marcenaria Construção - GRANULARIDADE RECUPERADA)
    ('44.18.10.JANELAS', rf'^(?!.*({EXCL_MATERIAIS}|{EXCL_DECORACAO}))(?=.*(JANELA|PORTA JANELA))(?=.*(MADEIRA|{TERMOS_ESPECIES})).*'),
    ('44.18.20.PORTAS', rf'^(?!.*({EXCL_MATERIAIS}|{EXCL_DECORACAO}))(?=.*(PORTA|FOLHA.*PORTA|BATENTE|CAIXILHO))(?!.*(JANELA)).*'),
    ('44.18.40.COFRAGEM', r'^(?=.*(COFRAGEM|FORMA.*CONCRETO))(?=.*(MADEIRA)).*'),
    ('44.18.7X.PISOS_ASSOALHOS', rf'^(?=.*(PISO|ASSOALHO|DECK|SOALHO|TACO))(?=.*(MADEIRA|BAMBU|{TERMOS_ESPECIES})).*'),
    ('44.18.9X.OUTROS_CONSTRUCAO', r'^(?=.*(VIGA|TRELIÇA|FORRO))(?=.*(CONSTRUCAO|OBRA))(?=.*(MADEIRA)).*'),

    # H. NCM 44.19 a 44.21 (Artefatos e Utensílios)
    ('44.19.UTENSILIOS_MESA', r'^(?=.*(PRATO|COPO|TIGELA|COLHER|BANDEJA|TABUA.*CARNE))(?=.*(MADEIRA|BAMBU)).*'),
    ('44.20.MARCHETARIA_ESTATUETAS', rf'^(?!.*({EXCL_DECORACAO}))(?=.*(MARCHETARIA|ESTATUETA|COFRE|CAIXA.*JOIA))(?=.*(MADEIRA)).*'),
    ('44.21.OUTRAS_OBRAS', rf'^(?!.*({EXCL_MATERIAIS}))(?=.*(CABIDE|PALITO|CEPO|CARRETEL|BOBINA|URNA|CERCA))(?=.*(MADEIRA|BAMBU)).*'),

    # I. CATEGORIA DE REVISÃO E FECHAMENTO
    ('AMBIGUOS_REVISAR', rf'.*(PLANEJADA|COZINHA|DALMOBILE|LOTE MISTO|MERCADORIA MISTA).*'),
    ('OUTROS', r'.*')
]

# ==============================================================================
# 3. PROCESSAMENTO E EXPORTAÇÃO
# ==============================================================================
def categorizar_produto(descricao):
    descricao_upper = str(descricao).upper()
    for categoria, regex in categorias_regex:
        if re.search(regex, descricao_upper):
            return categoria
    return 'NAO_CLASSIFICADO'

df_pandas['valor_total_vuntrib'] = df_pandas['prod_vuntrib'] * df_pandas['prod_qtrib'] * df_pandas['contagem']
df_pandas['categoria_regex'] = df_pandas['prod_xprod'].apply(categorizar_produto)

# Exportação Tripla
categorias_cap_44 = [c[0] for c in categorias_regex if c[0].startswith('44.') or c[0] == 'ROTULADOS ANTES']
categorias_eliminadas = ['OUTLIER__FINANCEIRO', 'OUTLIER__MAQUINAS', 'AMBIGUOS_REVISAR']

df_cap_44 = df_pandas[df_pandas['categoria_regex'].isin(categorias_cap_44)].copy()
df_outros = df_pandas[df_pandas['categoria_regex'] == 'OUTROS'].copy()
df_eliminados = df_pandas[df_pandas['categoria_regex'].isin(categorias_eliminadas)].copy()

df_cap_44.to_csv('base_treino_capitulo_44_pre.csv', sep=';', index=False, encoding='utf-8-sig')
df_outros.to_csv('base_outros_a_analisar_pre.csv', sep=';', index=False, encoding='utf-8-sig')
df_eliminados.to_csv('base_eliminados_lixo_pre.csv', sep=';', index=False, encoding='utf-8-sig')

print(f"1. CAPÍTULO 44 (Granular): {len(df_cap_44):>10,}".replace(",", "."))
print(f"2. OUTROS:                 {len(df_outros):>10,}".replace(",", "."))
print(f"3. ELIMINADOS:             {len(df_eliminados):>10,}".replace(",", "."))


In [ ]:
# Termos de exclusão para as regras complexas
termos_excl_janela = 'ALUMINIO|FERRO|METAL|INOX|PVC|VIDRO|BLINDEX|FUME|ACRILICO|PLASTICO|APLIQUE|RECORTE|LASER|MINIATURA|BONECA|POLLY|BRINQUEDO|ARTESANATO|ENFEITE|DECORACAO|FERRAGEM|FECHADURA|DOBRADICA|TRILHO|PUXADOR|CREMONA|ESPUMA|PU|VEDACAO|IMA|GELADEIRA|CORTINA|PERSIANA|TELA|MOSQUITEIRO|ADESIVO|LIMPEZA|PET|GATO|CACHORRO|ALIMENTADOR|ALBUM|FOTO|FICHARIO|HARD CASE|BAMBU|GIRASSOL|VASO|FLOR|PASSARO|CENARIO|ALTAR|CAPELA|ESPELHO|BASTIDOR|CACHEPO|PORTA RETRATO|JOIA|PRESENTE|ORGANIZADOR|TAMPO|FOGAO|MESA|CADEIRA'
termos_excl_carvao = 'ATIVADO|ATIVO|MINERAL|ANTRACIT|COQUE|HULHA|LINHITA|TURFA|FILTRO|MASCARA|SHAMPOO|SABONETE|CAPSULA|COMPRIMIDO|DENTIFRICO|PASTA DE DENTE'
termos_excl_palete = 'LOCACAO|REFORMA|ALUGUEL|CONSERTO|FRETE|MANUTENCAO|REPARO'
termos_excl_mdf = 'ARMARIO|ESTANTE|GAVETEIRO|MESA|CADEIRA|BALCAO|CRIADO|GUARDA ROUPA'
termos_ambiguos_exclusao = 'PLANEJADA|PLANEJADOS|COZINHA|DALMOBILE|EIXO|GUINDASTE|ELEVADOR|AUTOMATIC|ELETROMECANICO|ELETROHIDRAULICO|VW|DEERE|JOHN|CABINES|BOVINA|BOVINOS|CONTENCAO|PESCOCEIRAS|COIMMA|CAPUCCINO|FAVOS|IMPRESSO'
termos_ambiguos_inclusao = 'CABO|CABOS|FOLHA|FERRAMENTAS'

# Categorias REGEX
categorias_regex = [
    ('ROTULADOS ANTES', r'(LENHA|TORA DE MADEIRA|MADEIRA CERRADA|CABO DE MADEIRA|ARTIGO DOMESTICO DE MADEIRA|CARVAO VEGETAL|PORTA-BATENTE DE MADEIRA|PALITO DE DENTE|PALLET-EMBALAGEM DE MADEIRA|CHAPAS DE MDF|CHAPAS DE COMPENSADO|CHAPAS DE TAPUME|RESIDUO DE MADEIRA)'),
    ('OUTLIER__MAQUINAS', r'(PICADOR|TRATOR|COLHEITADEIRA|MÁQUINA|MAQUINA|VEÍCULO|VEICULO|PICADOR FLORESTAL)'),
    ('44.18.JANELA_MADEIRA', rf'^(?=.*JANELA)(?!.*({termos_excl_janela})).*'),
    ('44.02.CARVAO_VEGETAL', rf'^(?=.*(CARVÃO|CARVAO|MOINHA))(?!.*({termos_excl_carvao})).*'),
    ('44.21.PALETE_PALLET', rf'^(?=.*(PALETE|PALLET))(?!.*({termos_excl_palete})).*'),
    ('44.11.MDF_PAINEL', rf'^(?=.*(MDF|MDP|OSB|CHAPATEX|HDF))(?!.*({termos_excl_mdf})).*'),
    ('AMBIGUOS_EXCLUSAO', rf'.*({termos_ambiguos_exclusao}).*'),
    ('NOMES_AMBÍGUOS', rf'^(?=.*({termos_ambiguos_inclusao}))(?=.*(MADEIRA|MDF|MDP|PINUS|MM|X)).*'),
    ('44.03.MADEIRA_BRUTA', r'(MADEIRA.*(BRUTA|TORA|CELULOSE)|EUCALIPTO.*(TORA|CELULOSE)|PINUS.*BRUTA)'),
    ('44.18.FOLHA_PORTA', r'(FOLHA DE PORTA|PORTA|BATENTE|CAIXILHO|FORRO|SOALHO|PISO|ASSOALHO)'),
    ('44.01.CAVACO_RESIDUO', r'(CAVACO DE EUCALIPTO|LENHA|CAVACO|RESIDUO|GRANULADO|BIOMASSA|SERRAGEM|PO DE MADEIRA)'),
    ('44.21.OUTRAS_OBRAS', r'(PALITO|CEPO|ESTRADO|CAIXOTE)'),
    ('OUTROS', r'.*')
]

def categorizar_produto(descricao):
    descricao_upper = str(descricao).upper()
    for categoria, regex in categorias_regex:
        if re.search(regex, descricao_upper):
            return categoria
    return 'NAO_CLASSIFICADO'

df_pandas['valor_total_vuntrib'] = df_pandas['prod_vuntrib'] * df_pandas['prod_qtrib'] * df_pandas['contagem']
df_pandas['categoria_regex'] = df_pandas['prod_xprod'].apply(categorizar_produto)

In [ ]:
# Termos de exclusão para as regras complexas

# --- 1. DEFINIÇÃO DOS DICIONÁRIOS DE TERMOS (BASE PARA O REGEX) ---

# Exclusão absoluta (Serviços, Financeiro e Lixo de base)
termos_excl_prioritaria = 'ENTREGA|TAXA|FRETE|COMPLEMENTO DE VALOR|LOTE MISTO|MERCADORIA MISTA|SERVICO|VALOR ADICIONAL|AJUSTE'

# Materiais que invalidam o item (A menos que MADEIRA esteja presente)
termos_excl_materiais = 'VELUDO|PLASTICO|METAL|ALUMINIO|FERRO|ACO|VIDRO|BLINDEX|FUME|ACRILICO'

# Utensílios e Artefatos (Agora permitidos se validados por MADEIRA/BAMBU)
termos_artefatos_cap44 = 'CABIDE|ESPETO|ESPETINHO|CAKE BOARD|BANDEJA|CABO|FERRAMENTA|CAIXA|ADORNOS|PRENDEDOR|CAIXOTE|ESTRADO|PALITO|CEPO'

# Espécies Nobres e Validadas
termos_especies = 'CAMBARA|ANGELIM|TAUARI|CEDRINHO|CURUPIXA|ITAUBA|TATAJUBA|CUMARU|TECA|ABIURANA|CUPIUBA|AMESCLA|PINUS|EUCALIPTO|ARAUCO|GREENPLAC|CEDRO|MOGNO|VIROLA'

# Termos Técnicos e Processamento
termos_processamento = 'BENEFICIADA|SERRADA|SERRADO|VIGA|TABUA|TABUAS|TRONCO|FLORESTAL|CASCA|TORAS|TORA|CAVACO|LENHA|RESIDUO|BIOMASSA|MOINHA|18MM|1100X2200|FSC|CERTIFICADA|CERTIFICADO'

# Exclusões específicas de categorias
termos_excl_janela = 'INOX|PVC|APLIQUE|RECORTE|LASER|MINIATURA|BONECA|POLLY|BRINQUEDO|ARTESANATO|ENFEITE|DECORACAO|GELADEIRA|CORTINA|PERSIANA|TELA|MOSQUITEIRO'
termos_excl_carvao = 'ATIVADO|ATIVO|MINERAL|ANTRACIT|COQUE|HULHA|LINHITA|TURFA|FILTRO|MASCARA|SHAMPOO|SABONETE|CAPSULA|COMPRIMIDO'
termos_excl_mdf_moveis = 'ARMARIO|ESTANTE|GAVETEIRO|MESA|CADEIRA|BALCAO|CRIADO|GUARDA ROUPA|COZINHA|PLANEJADA|PLANEJADOS'


categorias_regex = [
    # A e B: Mantêm-se no topo (Bloqueio de lixo e máquinas)
    ('OUTLIER__FINANCEIRO', rf'.*({termos_excl_prioritaria}).*'),
    ('OUTLIER__MAQUINAS', r'(PICADOR|TRATOR|COLHEITADEIRA|MÁQUINA|MAQUINA|VEÍCULO|VEICULO|PICADOR FLORESTAL)'),
    
    # NOVIDADE: C. ESPÉCIES NOBRES E PROCESSAMENTO (SUBIU DE POSIÇÃO)
    # Se tem a espécie e termos de processamento, é Cap 44 com 99% de certeza.
    ('44.03.MADEIRA_BRUTA_SERRADA', rf'.*({termos_especies}).*({termos_processamento}).*'),

    # D. ROTULADOS ANTES
    ('ROTULADOS ANTES', r'(LENHA|TORA DE MADEIRA|MADEIRA CERRADA|CABO DE MADEIRA|ARTIGO DOMESTICO DE MADEIRA|CARVAO VEGETAL|PORTA-BATENTE DE MADEIRA|PALITO DE DENTE|PALLET-EMBALAGEM DE MADEIRA|CHAPAS DE MDF|CHAPAS DE COMPENSADO|CHAPAS DE TAPUME|RESIDUO DE MADEIRA)'),

    # E. CAP 44 - ARTEFATOS (Ajustado para aceitar espécies como validador)
    ('44.21.ARTEFATOS_MADEIRA', rf'^(?!.*({termos_excl_materiais})(?!.*(MADEIRA|MAD|MDF|BAMBU|{termos_especies})))(?=.*({termos_artefatos_cap44}))(?=.*(MADEIRA|MAD|MDF|BAMBU|CERTIFICADA|FSC|{termos_especies})).*'),

    # F. CAP 44 - PORTAS E JANELAS (Ajustado)
    ('44.18.PORTA_JANELA', rf'^(?!.*({termos_excl_materiais}|{termos_excl_janela})(?!.*(MADEIRA|MAD|MDF|{termos_especies})))(?=.*(PORTA|JANELA|BATENTE|CAIXILHO|FORRO|SOALHO|PISO|ASSOALHO))(?=.*(MADEIRA|MAD|MDF|{termos_especies}|210|80X|70X|90X)).*'),

    # G. CAP 44 - PAINÉIS
    ('44.11.MDF_PAINEL', rf'^(?!.*(PLASTIFICADO|{termos_excl_mdf_moveis}))(?=.*(MDF|MDP|OSB|HDF|COMPENSADO|ARAUCO|GREENPLAC))(?=.*(18MM|15MM|3MM|1100X2200|MADEIRA|MAD|{termos_especies})).*'),

    # H. CAP 44 - CARVÃO VEGETAL
    ('44.02.CARVAO_VEGETAL', rf'^(?=.*(CARVÃO|CARVAO|MOINHA))(?!.*({termos_excl_carvao})).*'),

    # I. CAP 44 - MATÉRIA-PRIMA REMANESCENTE (Caso não tenha pego na regra C)
    ('44.03.MADEIRA_BRUTA_SERRADA_GEN', rf'.*({termos_especies}|{termos_processamento}).*'),

    # J. PALETES
    ('44.21.PALETE_PALLET', r'^(?=.*(PALETE|PALLET))(?!.*(LOCACAO|ALUGUEL|REFORMA|FRETE)).*'),
    
    ('OUTROS', r'.*')
]


def categorizar_produto(descricao):
    descricao_upper = str(descricao).upper()
    for categoria, regex in categorias_regex:
        if re.search(regex, descricao_upper):
            return categoria
    return 'NAO_CLASSIFICADO'

df_pandas['valor_total_vuntrib'] = df_pandas['prod_vuntrib'] * df_pandas['prod_qtrib'] * df_pandas['contagem']
df_pandas['categoria_regex'] = df_pandas['prod_xprod'].apply(categorizar_produto)

In [ ]:
"""
REGEX COMPLETO - CAPÍTULO 44 NCM
Estrutura hierárquica baseada nas posições oficiais
"""

import re

# ============================================================================
# TERMOS DE EXCLUSÃO (aplicam-se a todas as categorias)
# ============================================================================

# Materiais que NÃO são madeira
EXCL_MATERIAIS = 'ALUMINIO|FERRO|METAL|INOX|PVC|VIDRO|PLASTICO|ACRILICO|CERAMICA|PORCELANA|CONCRETO|CIMENTO|GRANITO|MARMORE|PEDRA'

# Produtos prontos que vão para outros capítulos
EXCL_MOVEIS = 'ARMARIO|ESTANTE|MESA|CADEIRA|SOFA|POLTRONA|RACK|CRIADO MUDO|CAMA|BERCO|GUARDA ROUPA|COMODA|APARADOR|BUFFET|CRISTALEIRA'

# Objetos decorativos/brinquedos
EXCL_DECORACAO = 'BRINQUEDO|BONECA|MINIATURA|ARTESANATO|ENFEITE|JOIA|BIJUTERIA'

# Serviços
EXCL_SERVICOS = 'LOCACAO|ALUGUEL|REFORMA|CONSERTO|MANUTENCAO|REPARO|FRETE|MONTAGEM|INSTALACAO'

# Outros produtos ambíguos
EXCL_OUTROS = 'CARVAO ATIVADO|CARVAO MINERAL|ANTRACIT|COQUE|HULHA|TURFA|FILTRO|MASCARA'

# Consolidado
TERMOS_EXCLUSAO_GERAIS = f'{EXCL_MATERIAIS}|{EXCL_MOVEIS}|{EXCL_DECORACAO}|{EXCL_SERVICOS}|{EXCL_OUTROS}'

# ============================================================================
# 44.01 - LENHA, ESTILHAS, SERRAGEM E RESÍDUOS
# ============================================================================

REGEX_4401 = {
    'codigo': '44.01',
    'nome': 'LENHA_RESIDUOS_SERRAGEM',
    'termos_positivos': 'LENHA|TORO|TORA|ACHA|GRAVETO|FEIXE|ESTILHA|PARTICULA|SERRAGEM|SERRADURA|DESPERDICIO|RESIDUO|CAVACO|RASPA|APARAS|PO DE MADEIRA|BRIQUETE|PELLET|BIOMASSA|GRANULADO',
    'termos_contexto': 'MADEIRA|EUCALIPTO|PINUS|VEGETAL',  # Pelo menos um desses
    'exclusoes': TERMOS_EXCLUSAO_GERAIS,
    'regex': r'^(?=.*(LENHA|TORO|TORA|ACHA|GRAVETO|FEIXE|ESTILHA|PARTICULA|SERRAGEM|SERRADURA|DESPERDICIO|RESIDUO|CAVACO|RASPA|APARAS|PO DE MADEIRA|BRIQUETE|PELLET|BIOMASSA|GRANULADO))(?!.*(ALUMINIO|FERRO|METAL|INOX|PVC|VIDRO|PLASTICO|ACRILICO|CARVAO ATIVADO|CARVAO MINERAL)).*'
}

# ============================================================================
# 44.02 - CARVÃO VEGETAL
# ============================================================================

REGEX_4402 = {
    'codigo': '44.02',
    'nome': 'CARVAO_VEGETAL',
    'termos_positivos': 'CARVAO|CARVÃO|MOINHA',
    'termos_contexto': 'VEGETAL|MADEIRA|EUCALIPTO',
    'exclusoes': 'ATIVADO|ATIVO|MINERAL|ANTRACIT|COQUE|HULHA|LINHITA|TURFA|FILTRO|MASCARA|SHAMPOO|SABONETE|CAPSULA|COMPRIMIDO|DENTIFRICO|PASTA DE DENTE',
    'regex': r'^(?=.*(CARVAO|CARVÃO|MOINHA))(?!.*(ATIVADO|ATIVO|MINERAL|ANTRACIT|COQUE|HULHA|LINHITA|TURFA|FILTRO|MASCARA|SHAMPOO|SABONETE|CAPSULA|COMPRIMIDO|DENTIFRICO)).*'
}

# ============================================================================
# 44.03 - MADEIRA EM BRUTO
# ============================================================================

REGEX_4403 = {
    'codigo': '44.03',
    'nome': 'MADEIRA_BRUTA',
    'termos_positivos': 'MADEIRA BRUTA|MADEIRA EM BRUTO|TORA|TORO|RONCO|DESCASCADA|DESALBURNADA|ESQUADRIADA',
    'termos_tipos': 'PINUS|EUCALIPTO|TECA|MOGNO|CEDRO|IMBUIA|IPE|PEROBA|ARAUCARIA|CONÍFERA',
    'termos_tratamento': 'TRATADA|CREOSOTO|CONSERVACAO|PRESERVATIVO|AUTOCLAVE',
    'exclusoes': f'{EXCL_MATERIAIS}|SERRADA|APARELHADA|BENEFICIADA',
    'regex': r'^(?=.*(MADEIRA.*(BRUTA|EM BRUTO)|TORA|TORO|RONCO))(?!.*(ALUMINIO|FERRO|METAL|SERRADA|APARELHADA|BENEFICIADA)).*'
}

# ============================================================================
# 44.04 - ARCOS, ESTACAS E MADEIRA DESBASTADA
# ============================================================================

REGEX_4404 = {
    'codigo': '44.04',
    'nome': 'ARCOS_ESTACAS',
    'termos_positivos': 'ARCO DE MADEIRA|ESTACA|MOIRAO|POSTE ROLICO|MADEIRA DESBASTADA|MADEIRA ARREDONDADA',
    'exclusoes': f'{EXCL_MATERIAIS}|SERRADA|APARELHADA',
    'regex': r'^(?=.*(ARCO DE MADEIRA|ESTACA|MOIRAO|POSTE ROLICO|DESBASTADA|ARREDONDADA))(?!.*(METAL|CONCRETO|SERRADA)).*'
}

# ============================================================================
# 44.05 - LÃ E FARINHA DE MADEIRA
# ============================================================================

REGEX_4405 = {
    'codigo': '44.05',
    'nome': 'LA_FARINHA_MADEIRA',
    'termos_positivos': 'LA DE MADEIRA|FARINHA DE MADEIRA|FIBRA DE MADEIRA',
    'exclusoes': EXCL_MATERIAIS,
    'regex': r'^(?=.*(LA DE MADEIRA|FARINHA DE MADEIRA|FIBRA DE MADEIRA))(?!.*(MINERAL|VIDRO|SINTETICA)).*'
}

# ============================================================================
# 44.06 - DORMENTES PARA VIAS FÉRREAS
# ============================================================================

REGEX_4406 = {
    'codigo': '44.06',
    'nome': 'DORMENTES_FERROVIARIOS',
    'termos_positivos': 'DORMENTE|TRILHO|FERROVIA|FERREO|VIA FERREA',
    'termos_contexto': 'MADEIRA',
    'exclusoes': f'{EXCL_MATERIAIS}|CONCRETO',
    'regex': r'^(?=.*(DORMENTE))(?=.*(MADEIRA|FERROVIA|TRILHO))(?!.*(CONCRETO|METAL)).*'
}

# ============================================================================
# 44.07 - MADEIRA SERRADA OU FENDIDA (> 6mm)
# ============================================================================

REGEX_4407 = {
    'codigo': '44.07',
    'nome': 'MADEIRA_SERRADA',
    'termos_positivos': 'MADEIRA SERRADA|TABUA|PRANCHA|VIGA|VIGOTA|CAIBRO|RIPA|SARRAFO|BARROTE|PONTALETE',
    'termos_tipos': 'PINUS|EUCALIPTO|PEROBA|CEDRO|MOGNO|IPE|JATOBA|ANGELIM|CUMARU',
    'termos_beneficiamento': 'APLAINADA|PLAINA|APARELHADA|SECA|KD|TRATADA|AUTOCLAVE',
    'exclusoes': f'{EXCL_MATERIAIS}|COMPENSADO|MDF|PAINEL|CHAPA',
    'regex': r'^(?=.*(MADEIRA SERRADA|TABUA|PRANCHA|VIGA|VIGOTA|CAIBRO|RIPA|SARRAFO|BARROTE|PONTALETE))(?!.*(METAL|COMPENSADO|MDF|PAINEL)).*'
}

# ============================================================================
# 44.08 - FOLHAS PARA FOLHEADOS (≤ 6mm)
# ============================================================================

REGEX_4408 = {
    'codigo': '44.08',
    'nome': 'FOLHAS_FOLHEADOS',
    'termos_positivos': 'FOLHA PARA FOLHEADO|FOLHEADO|LAMINA|DESENROLADA',
    'termos_espessura': '6MM|6 MM|FINA|DELGADA',
    'exclusoes': f'{EXCL_MATERIAIS}|COMPENSADO PRONTO',
    'regex': r'^(?=.*(FOLHA.*FOLHEADO|FOLHEADO|LAMINA.*MADEIRA))(?=.*(6MM|FINA|DELGADA))(?!.*(METAL|PLASTICO)).*'
}

# ============================================================================
# 44.09 - MADEIRA PERFILADA
# ============================================================================

REGEX_4409 = {
    'codigo': '44.09',
    'nome': 'MADEIRA_PERFILADA',
    'termos_positivos': 'FASQUIA|FRISADO|FORRO|LAMBRI|DECK|ASSOALHO|SOALHO|PISO DE MADEIRA|TACO|RODAPE|BAGUETE|MOLDURA LISA',
    'termos_perfil': 'MACHO E FEMEA|MACHO FEMEA|ENCAIXE|PERFILADA|RANHURA|CHANFRO',
    'exclusoes': f'{EXCL_MATERIAIS}|{EXCL_MOVEIS}|PORTA|JANELA',
    'regex': r'^(?=.*(FASQUIA|FRISADO|FORRO|LAMBRI|DECK|ASSOALHO|SOALHO|PISO.*MADEIRA|TACO|RODAPE|BAGUETE))(?!.*(PLASTICO|PVC|CERAMICA|PORCELANATO)).*'
}

# ============================================================================
# 44.10 - PAINÉIS DE PARTÍCULAS (OSB, MDP, AGLOMERADO)
# ============================================================================

REGEX_4410 = {
    'codigo': '44.10',
    'nome': 'PAINEIS_PARTICULAS',
    'termos_positivos': 'OSB|MDP|AGLOMERADO|PAINEL DE PARTICULA|CHAPA DE PARTICULA|ORIENTED STRAND BOARD',
    'termos_medidas': 'MM|X|ESPESSURA|CHAPA|PAINEL',
    'exclusoes': f'{EXCL_MOVEIS}|MDF|COMPENSADO|HDF',
    'regex': r'^(?=.*(OSB|MDP|AGLOMERADO|PAINEL.*PARTICULA))(?!.*(ARMARIO|MESA|CADEIRA|ESTANTE)).*'
}

# ============================================================================
# 44.11 - PAINÉIS DE FIBRAS (MDF, HDF)
# ============================================================================

REGEX_4411 = {
    'codigo': '44.11',
    'nome': 'PAINEIS_FIBRAS_MDF',
    'termos_positivos': 'MDF|HDF|CHAPATEX|EUCATEX|PAINEL DE FIBRA|CHAPA DE FIBRA|FIBRA DE MADEIRA',
    'termos_densidade': 'MEDIA DENSIDADE|ALTA DENSIDADE|BAIXA DENSIDADE',
    'termos_medidas': 'MM|X|ESPESSURA|CHAPA|PAINEL',
    'exclusoes': f'{EXCL_MOVEIS}|OSB|MDP|COMPENSADO',
    'regex': r'^(?=.*(MDF|HDF|CHAPATEX|EUCATEX|PAINEL.*FIBRA))(?!.*(ARMARIO|MESA|CADEIRA|ESTANTE|GUARDA ROUPA)).*'
}

# ============================================================================
# 44.12 - MADEIRA COMPENSADA (CONTRAPLACADA)
# ============================================================================

REGEX_4412 = {
    'codigo': '44.12',
    'nome': 'COMPENSADO_CONTRAPLACADO',
    'termos_positivos': 'COMPENSADO|COMPENSADA|CONTRAPLACADO|LAMINADO|PLYWOOD|MADEIRA ESTRATIFICADA',
    'termos_tipos': 'NAVAL|RESINADO|PLASTIFICADO|ESTRUTURAL|SARRAFEADO',
    'termos_madeira': 'PINUS|EUCALIPTO|VIROLA|TROPICAL',
    'termos_medidas': 'MM|X|ESPESSURA|CHAPA|FOLHA',
    'exclusoes': f'{EXCL_MOVEIS}|MDF|OSB|PORTA PRONTA|JANELA PRONTA',
    'regex': r'^(?=.*(COMPENSADO|COMPENSADA|CONTRAPLACADO|LAMINADO|PLYWOOD))(?!.*(ARMARIO|MESA|CADEIRA|PORTA PRONTA|JANELA PRONTA)).*'
}

# ============================================================================
# 44.13 - MADEIRA DENSIFICADA
# ============================================================================

REGEX_4413 = {
    'codigo': '44.13',
    'nome': 'MADEIRA_DENSIFICADA',
    'termos_positivos': 'MADEIRA DENSIFICADA|DENSIFIED WOOD|MADEIRA COMPRIMIDA',
    'exclusoes': EXCL_MATERIAIS,
    'regex': r'^(?=.*(MADEIRA DENSIFICADA|DENSIFIED WOOD|MADEIRA COMPRIMIDA)).*'
}

# ============================================================================
# 44.14 - MOLDURAS PARA QUADROS
# ============================================================================

REGEX_4414 = {
    'codigo': '44.14',
    'nome': 'MOLDURAS_QUADROS',
    'termos_positivos': 'MOLDURA|PORTA RETRATO|QUADRO|ESPELHO|BAGUETE',
    'termos_contexto': 'MADEIRA',
    'exclusoes': f'{EXCL_MATERIAIS}|FOTO|POSTER|TELA',
    'regex': r'^(?=.*(MOLDURA|PORTA RETRATO))(?=.*(MADEIRA))(?!.*(PLASTICO|METAL|FOTO|POSTER)).*'
}

# ============================================================================
# 44.15 - CAIXOTES, PALETES E EMBALAGENS
# ============================================================================

REGEX_4415 = {
    'codigo': '44.15',
    'nome': 'CAIXOTES_PALETES_EMBALAGENS',
    'termos_positivos': 'PALETE|PALLET|ESTRADO|CAIXOTE|CAIXA DE MADEIRA|ENGRADADO|BARRICA|CARRETEL|BOBINA',
    'termos_tipos': 'PBR|DESCARTAVEL|EXPORTACAO|EMBALAGEM',
    'exclusoes': f'{EXCL_SERVICOS}|LOCACAO|REFORMA',
    'regex': r'^(?=.*(PALETE|PALLET|ESTRADO|CAIXOTE|CAIXA.*MADEIRA|ENGRADADO|BARRICA|CARRETEL))(?!.*(LOCACAO|REFORMA|ALUGUEL|CONSERTO)).*'
}

# ============================================================================
# 44.16 - BARRIS E OBRAS DE TANOEIRO
# ============================================================================

REGEX_4416 = {
    'codigo': '44.16',
    'nome': 'BARRIS_TANOEIRO',
    'termos_positivos': 'BARRIL|TONEL|PIPA|CUBA|DORNAS|TANOARIA|ADUELA',
    'termos_contexto': 'MADEIRA|CARVALHO|CACHACA|VINHO',
    'exclusoes': f'{EXCL_MATERIAIS}|INOX',
    'regex': r'^(?=.*(BARRIL|TONEL|PIPA|CUBA|DORNAS|TANOARIA|ADUELA))(?=.*(MADEIRA))(?!.*(METAL|INOX|PLASTICO)).*'
}

# ============================================================================
# 44.17 - FERRAMENTAS E CABOS DE MADEIRA
# ============================================================================

REGEX_4417 = {
    'codigo': '44.17',
    'nome': 'FERRAMENTAS_CABOS',
    'termos_positivos': 'CABO DE MADEIRA|CABO PARA|VASSOURA|RODO|PA|ENXADA|PICARETA|MARTELO|MACHADO|FERRAMENTA',
    'termos_contexto': 'MADEIRA',
    'exclusoes': f'{EXCL_MATERIAIS}|FIBRA|SINTETICO',
    'regex': r'^(?=.*(CABO.*MADEIRA|CABO PARA|VASSOURA|RODO))(?!.*(PLASTICO|FIBRA|SINTETICO|METAL)).*'
}

# ============================================================================
# 44.18 - OBRAS DE MARCENARIA PARA CONSTRUÇÃO
# ============================================================================

REGEX_4418 = {
    'codigo': '44.18',
    'nome': 'OBRAS_MARCENARIA_CONSTRUCAO',
    'termos_positivos': 'PORTA|FOLHA DE PORTA|BATENTE|MARCO|ALIZARES|CAIXILHO|JANELA|PORTA JANELA|VENEZIANA|ESQUADRIA|GUARNIÇÃO|TRAVESSA|MONTANTE|SOLEIRA|PAINEL CELULAR|SHINGLE|SHAKE|COFRAGEM',
    'termos_materiais': 'MADEIRA|COMPENSADO|MDF|PINUS|CEDRO',
    'exclusoes': f'{EXCL_MATERIAIS}|ALUMINIO|PVC|FERRO|METAL|FECHADURA|DOBRADICA|FERRAGEM',
    'regex': r'^(?=.*(PORTA|FOLHA DE PORTA|BATENTE|MARCO|JANELA|PORTA JANELA|CAIXILHO|VENEZIANA|ESQUADRIA))(?!.*(ALUMINIO|FERRO|METAL|PVC|VIDRO|FECHADURA|DOBRADICA|FERRAGEM|PUXADOR)).*'
}

# Sub-categorias específicas 44.18
REGEX_4418_PORTAS = {
    'codigo': '44.18.20',
    'nome': 'PORTAS_FOLHAS',
    'regex': r'^(?=.*(PORTA|FOLHA.*PORTA|FOLHA DE PORTA))(?!.*(ALUMINIO|FERRO|METAL|PVC|VIDRO|FECHADURA|DOBRADICA|FERRAGEM)).*'
}

REGEX_4418_JANELAS = {
    'codigo': '44.18.10',
    'nome': 'JANELAS_PORTA_JANELAS',
    'regex': r'^(?=.*(JANELA|PORTA JANELA|PORTA-JANELA))(?!.*(ALUMINIO|FERRO|METAL|PVC|VIDRO|BLINDEX)).*'
}

# ============================================================================
# 44.19 - ARTEFATOS DE MESA E COZINHA
# ============================================================================

REGEX_4419 = {
    'codigo': '44.19',
    'nome': 'ARTEFATOS_MESA_COZINHA',
    'termos_positivos': 'TABUA|TABUA DE CARNE|COLHER DE PAU|ESPATULA|UTENSILIOS|SALEIRO|PIMENTEIRO|PETISQUEIRA|TIGELA|BOWL',
    'termos_contexto': 'MADEIRA|BAMBU',
    'exclusoes': f'{EXCL_MATERIAIS}|{EXCL_MOVEIS}|MESA|CADEIRA',
    'regex': r'^(?=.*(TABUA|COLHER DE PAU|ESPATULA|UTENSILIOS|SALEIRO|PIMENTEIRO))(?=.*(MADEIRA|BAMBU))(?!.*(PLASTICO|VIDRO)).*'
}

# ============================================================================
# 44.20 - MARCHETARIA E ESTATUETAS
# ============================================================================

REGEX_4420 = {
    'codigo': '44.20',
    'nome': 'MARCHETARIA_ESTATUETAS',
    'termos_positivos': 'MARCHETARIA|EMBUTIDO|INTARSIA|ESTATUETA|ESCULTURA|ENTALHE|ESTOJO|PORTA JOIA|CAIXINHA',
    'termos_contexto': 'MADEIRA',
    'exclusoes': f'{EXCL_MATERIAIS}|{EXCL_MOVEIS}|BRINQUEDO',
    'regex': r'^(?=.*(MARCHETARIA|EMBUTIDO|ESTATUETA|ESCULTURA|ENTALHE|ESTOJO|PORTA JOIA))(?=.*(MADEIRA))(?!.*(BRINQUEDO|MINIATURA)).*'
}

# ============================================================================
# 44.21 - OUTRAS OBRAS DE MADEIRA
# ============================================================================

REGEX_4421 = {
    'codigo': '44.21',
    'nome': 'OUTRAS_OBRAS_MADEIRA',
    'termos_positivos': 'CABIDE|PALITO|PALITO DE DENTE|ABAIXADOR DE LINGUA|ESPATULA MEDICA|CARRETEL|BOBINA|CANELA|CAIXAO|URNA|ESQUIFE|CERCA|COLMEIA|GAIOLA|VARAL',
    'termos_contexto': 'MADEIRA',
    'exclusoes': f'{EXCL_MATERIAIS}|{EXCL_MOVEIS}',
    'regex': r'^(?=.*(CABIDE|PALITO|ABAIXADOR|ESPATULA MEDICA|CARRETEL|BOBINA|CAIXAO|URNA|CERCA|COLMEIA|GAIOLA))(?!.*(PLASTICO|METAL)).*'
}

# ============================================================================
# SUB-ITENS IMPORTANTES (6-8 DÍGITOS)
# ============================================================================

SUB_ITENS_DETALHADOS = {
    # 4403 - Madeira em bruto
    '4403.11': {
        'nome': 'PINUS_TRATADO',
        'regex': r'^(?=.*(PINUS|CONÍFERA))(?=.*(TRATAD|CREOSOT|AUTOCLAVE|CCA|CCB))(?=.*(BRUT|TORA|RONCO)).*'
    },
    '4403.12': {
        'nome': 'PINUS_NAO_TRATADO',
        'regex': r'^(?=.*(PINUS|CONÍFERA))(?!.*(TRATAD|CREOSOT|AUTOCLAVE))(?=.*(BRUT|TORA|RONCO)).*'
    },
    '4403.21': {
        'nome': 'EUCALIPTO_TRATADO',
        'regex': r'^(?=.*(EUCALIPT))(?=.*(TRATAD|AUTOCLAVE|CCA|CCB))(?=.*(BRUT|TORA|RONCO)).*'
    },
    '4403.22': {
        'nome': 'EUCALIPTO_NAO_TRATADO',
        'regex': r'^(?=.*(EUCALIPT))(?!.*(TRATAD|AUTOCLAVE))(?=.*(BRUT|TORA|RONCO)).*'
    },
    
    # 4407 - Madeira serrada
    '4407.11': {
        'nome': 'PINUS_SERRADO',
        'regex': r'^(?=.*(PINUS))(?=.*(SERRAD|TABUA|PRANCHA|VIGA|CAIBRO|RIPA|SARRAFO)).*'
    },
    '4407.12': {
        'nome': 'PINUS_APLAINADO',
        'regex': r'^(?=.*(PINUS))(?=.*(APLAINAD|PLAINA|APARELHAD)).*'
    },
    '4407.29': {
        'nome': 'EUCALIPTO_SERRADO',
        'regex': r'^(?=.*(EUCALIPT))(?=.*(SERRAD|TABUA|PRANCHA|VIGA)).*'
    },
    
    # 4410 - Painéis de partículas
    '4410.11': {
        'nome': 'OSB',
        'regex': r'^(?=.*(OSB|ORIENTED STRAND)).*'
    },
    '4410.12': {
        'nome': 'MDP_AGLOMERADO',
        'regex': r'^(?=.*(MDP|AGLOMERADO))(?!.*(MDF)).*'
    },
    
    # 4411 - Painéis de fibras (MDF/HDF)
    '4411.12': {
        'nome': 'MDF',
        'regex': r'^(?=.*(MDF|MEDIA DENSIDADE))(?!.*(HDF)).*'
    },
    '4411.13': {
        'nome': 'HDF',
        'regex': r'^(?=.*(HDF|ALTA DENSIDADE)).*'
    },
    '4411.14': {
        'nome': 'FIBRA_BAIXA_DENSIDADE',
        'regex': r'^(?=.*(FIBRA|CHAPATEX))(?=.*(BAIXA DENSIDADE)).*'
    },
    
    # 4412 - Compensados
    '4412.31': {
        'nome': 'COMPENSADO_TROPICAL',
        'regex': r'^(?=.*(COMPENSAD|CONTRAPLACAD))(?=.*(TROPICAL|VIROLA|MOGNO)).*'
    },
    '4412.33': {
        'nome': 'COMPENSADO_PINUS',
        'regex': r'^(?=.*(COMPENSAD|CONTRAPLACAD))(?=.*(PINUS)).*'
    },
    '4412.94': {
        'nome': 'COMPENSADO_OUTROS',
        'regex': r'^(?=.*(COMPENSAD|CONTRAPLACAD|LAMINAD))(?!.*(TROPICAL|PINUS)).*'
    },
    
    # 4415 - Embalagens
    '4415.10': {
        'nome': 'CAIXOTES_EMBALAGENS',
        'regex': r'^(?=.*(CAIXOTE|CAIXA|ENGRADADO))(?=.*(MADEIRA))(?!.*(PALETE|PALLET)).*'
    },
    '4415.20': {
        'nome': 'PALETES',
        'regex': r'^(?=.*(PALETE|PALLET|ESTRADO))(?!.*(LOCACAO|ALUGUEL|REFORMA)).*'
    },
    
    # 4418 - Marcenaria construção
    '4418.10': {
        'nome': 'JANELAS_MADEIRA',
        'regex': r'^(?=.*(JANELA|PORTA JANELA))(?!.*(ALUMINIO|PVC|METAL)).*'
    },
    '4418.20': {
        'nome': 'PORTAS_MADEIRA',
        'regex': r'^(?=.*(PORTA|FOLHA.*PORTA|BATENTE))(?!.*(ALUMINIO|PVC|METAL|JANELA)).*'
    },
    '4418.40': {
        'nome': 'COFRAGEM',
        'regex': r'^(?=.*(COFRAGEM|FORMA.*CONCRETO))(?=.*(MADEIRA)).*'
    },
    '4418.73': {
        'nome': 'PISO_BAMBU',
        'regex': r'^(?=.*(PISO|ASSOALHO|DECK))(?=.*(BAMBU)).*'
    },
    '4418.74': {
        'nome': 'PISO_MADEIRA',
        'regex': r'^(?=.*(PISO|ASSOALHO|DECK|SOALHO|TACO))(?=.*(MADEIRA|PINUS|EUCALIPTO|IPE|CUMARU)).*'
    },
    
    # 4421 - Outras obras
    '4421.10': {
        'nome': 'CABIDES',
        'regex': r'^(?=.*(CABIDE))(?=.*(MADEIRA)).*'
    },
    '4421.91': {
        'nome': 'PALITOS',
        'regex': r'^(?=.*(PALITO|ABAIXADOR))(?!.*(FERRO|METAL)).*'
    },
    '4421.99': {
        'nome': 'OUTRAS_OBRAS',
        'regex': r'^(?=.*(CARRETEL|BOBINA|CANELA|CAIXAO|URNA|CERCA|COLMEIA))(?=.*(MADEIRA)).*'
    }
}

# ============================================================================
# LISTA CONSOLIDADA DE TODAS AS CATEGORIAS
# ============================================================================

CATEGORIAS_CAP44_COMPLETAS = [
    # Posições principais (4 dígitos)
    ('44.01.LENHA_RESIDUOS', REGEX_4401['regex'], True),
    ('44.02.CARVAO_VEGETAL', REGEX_4402['regex'], True),
    ('44.03.MADEIRA_BRUTA', REGEX_4403['regex'], True),
    ('44.04.ARCOS_ESTACAS', REGEX_4404['regex'], True),
    ('44.05.LA_FARINHA', REGEX_4405['regex'], True),
    ('44.06.DORMENTES', REGEX_4406['regex'], True),
    ('44.07.MADEIRA_SERRADA', REGEX_4407['regex'], True),
    ('44.08.FOLHAS_FOLHEADOS', REGEX_4408['regex'], True),
    ('44.09.MADEIRA_PERFILADA', REGEX_4409['regex'], True),
    ('44.10.PAINEIS_PARTICULAS', REGEX_4410['regex'], True),
    ('44.11.PAINEIS_FIBRAS_MDF', REGEX_4411['regex'], True),
    ('44.12.COMPENSADO', REGEX_4412['regex'], True),
    ('44.13.MADEIRA_DENSIFICADA', REGEX_4413['regex'], True),
    ('44.14.MOLDURAS', REGEX_4414['regex'], True),
    ('44.15.CAIXOTES_PALETES', REGEX_4415['regex'], True),
    ('44.16.BARRIS_TANOEIRO', REGEX_4416['regex'], True),
    ('44.17.FERRAMENTAS_CABOS', REGEX_4417['regex'], True),
    ('44.18.MARCENARIA_CONSTRUCAO', REGEX_4418['regex'], True),
    ('44.19.ARTEFATOS_MESA', REGEX_4419['regex'], True),
    ('44.20.MARCHETARIA', REGEX_4420['regex'], True),
    ('44.21.OUTRAS_OBRAS', REGEX_4421['regex'], True),
]

# 1. FUNÇÃO CORRIGIDA
def categorizar_produto(descricao):
    descricao_upper = str(descricao).upper()
    for categoria, regex_pattern, is_cap44 in CATEGORIAS_CAP44_COMPLETAS:
        if re.search(regex_pattern, descricao_upper):
            return categoria
    return 'NAO_CLASSIFICADO'

# 2. APLICAR NO DATAFRAME
df_pandas['valor_total_vuntrib'] = (
    df_pandas['prod_vuntrib'] * 
    df_pandas['prod_qtrib'] * 
    df_pandas['contagem']
)

df_pandas['categoria_regex'] = df_pandas['prod_xprod'].apply(categorizar_produto)

# 3. ANÁLISE POR CONTAGEM
contagem_absoluta = df_pandas['categoria_regex'].value_counts()
total_linhas = len(df_pandas)
representatividade_contagem_pct = (contagem_absoluta / total_linhas) * 100

print("REPRESENTATIVIDADE POR CONTAGEM:")
print(representatividade_contagem_pct.head(20).round(2))

# 4. ANÁLISE FINANCEIRA
valor_por_categoria = df_pandas.groupby('categoria_regex')['valor_total_vuntrib'].sum()
total_valor = df_pandas['valor_total_vuntrib'].sum()
representatividade_percentual = (valor_por_categoria / total_valor) * 100

print("\nREPRESENTATIVIDADE FINANCEIRA:")
print(representatividade_percentual.sort_values(ascending=False).head(20).round(2))

# 5. COMPARATIVO
comparativo = pd.DataFrame({
    'Volume (%)': representatividade_contagem_pct,
    'Financeiro (%)': representatividade_percentual
})

print("\nCOMPARATIVO:")
print(comparativo.head(20).round(2))

In [ ]:
"""
ANÁLISE COMPLETA - Cap 44 com Nova Estrutura de Categorias
Adaptado para CATEGORIAS_CAP44_COMPLETAS
"""

import pandas as pd
import numpy as np

# ============================================================================
# DEFINIÇÃO DAS CATEGORIAS
# ============================================================================

# Todas as categorias que PERTENCEM ao Capítulo 44
# (extraídas automaticamente de CATEGORIAS_CAP44_COMPLETAS onde is_cap44=True)
CATEGORIAS_CAP_44 = [
    '44.01.LENHA_RESIDUOS',
    '44.02.CARVAO_VEGETAL',
    '44.03.MADEIRA_BRUTA',
    '44.04.ARCOS_ESTACAS',
    '44.05.LA_FARINHA',
    '44.06.DORMENTES',
    '44.07.MADEIRA_SERRADA',
    '44.08.FOLHAS_FOLHEADOS',
    '44.09.MADEIRA_PERFILADA',
    '44.10.PAINEIS_PARTICULAS',
    '44.11.PAINEIS_FIBRAS_MDF',
    '44.12.COMPENSADO',
    '44.13.MADEIRA_DENSIFICADA',
    '44.14.MOLDURAS',
    '44.15.CAIXOTES_PALETES',
    '44.16.BARRIS_TANOEIRO',
    '44.17.FERRAMENTAS_CABOS',
    '44.18.MARCENARIA_CONSTRUCAO',
    '44.19.ARTEFATOS_MESA',
    '44.20.MARCHETARIA',
    '44.21.OUTRAS_OBRAS',
]

# Categorias que foram explicitamente excluídas (NÃO são Cap 44)
# Essas seriam adicionadas se você criar categorias de exclusão
CATEGORIAS_ELIMINADAS = [
    # Adicione aqui se criar categorias de exclusão explícitas
    # Por exemplo: 'OUTLIER__MAQUINAS', 'AMBIGUOS_EXCLUSAO', etc.
]

# ============================================================================
# FUNÇÃO PARA EXTRAIR AUTOMATICAMENTE AS CATEGORIAS
# ============================================================================

def extrair_categorias_automaticamente():
    """
    Extrai categorias Cap 44 automaticamente de CATEGORIAS_CAP44_COMPLETAS
    """
    categorias_cap44 = []
    
    for categoria, regex, is_cap44 in CATEGORIAS_CAP44_COMPLETAS:
        if is_cap44:
            categorias_cap44.append(categoria)
    
    return categorias_cap44

# ============================================================================
# ANÁLISE DE REPRESENTATIVIDADE COMPLETA
# ============================================================================

def analisar_representatividade_completa(df_pandas):
    """
    Análise completa de representatividade do Cap 44
    
    Divide o dataset em:
    1. CAPÍTULO 44: Itens classificados como Cap 44
    2. OUTROS: Itens não classificados
    3. ELIMINADOS: Itens explicitamente excluídos (se houver)
    """
    
    print("="*100)
    print("ANÁLISE DE REPRESENTATIVIDADE - CAPÍTULO 44")
    print("="*100)
    
    # Verificar se coluna categoria_regex existe
    if 'categoria_regex' not in df_pandas.columns:
        print("\n❌ ERRO: Coluna 'categoria_regex' não encontrada!")
        print("Execute primeiro: df_pandas['categoria_regex'] = df_pandas['prod_xprod'].apply(categorizar_produto)")
        return None
    
    # 1. Separar datasets
    print("\n→ Separando datasets por categoria...")
    
    df_cap_44 = df_pandas[df_pandas['categoria_regex'].isin(CATEGORIAS_CAP_44)].copy()
    df_outros = df_pandas[df_pandas['categoria_regex'] == 'NAO_CLASSIFICADO'].copy()
    df_eliminados = df_pandas[df_pandas['categoria_regex'].isin(CATEGORIAS_ELIMINADAS)].copy()
    
    # Calcular valor total se não existir
    if 'valor_total_vuntrib' not in df_pandas.columns:
        print("→ Calculando valores totais...")
        if 'contagem' in df_pandas.columns:
            df_pandas['valor_total_vuntrib'] = (
                df_pandas['prod_vuntrib'] * 
                df_pandas['prod_qtrib'] * 
                df_pandas['contagem']
            )
            df_cap_44['valor_total_vuntrib'] = (
                df_cap_44['prod_vuntrib'] * 
                df_cap_44['prod_qtrib'] * 
                df_cap_44['contagem']
            )
            df_outros['valor_total_vuntrib'] = (
                df_outros['prod_vuntrib'] * 
                df_outros['prod_qtrib'] * 
                df_outros['contagem']
            )
            if len(df_eliminados) > 0:
                df_eliminados['valor_total_vuntrib'] = (
                    df_eliminados['prod_vuntrib'] * 
                    df_eliminados['prod_qtrib'] * 
                    df_eliminados['contagem']
                )
        else:
            df_pandas['valor_total_vuntrib'] = df_pandas['prod_vuntrib'] * df_pandas['prod_qtrib']
            df_cap_44['valor_total_vuntrib'] = df_cap_44['prod_vuntrib'] * df_cap_44['prod_qtrib']
            df_outros['valor_total_vuntrib'] = df_outros['prod_vuntrib'] * df_outros['prod_qtrib']
            if len(df_eliminados) > 0:
                df_eliminados['valor_total_vuntrib'] = df_eliminados['prod_vuntrib'] * df_eliminados['prod_qtrib']
    
    # 2. ESTATÍSTICAS GERAIS
    print("\n" + "="*100)
    print("1. RESUMO GERAL - VOLUME (Quantidade de Registros)")
    print("="*100)
    
    total_registros = len(df_pandas)
    
    print(f"\n{'Categoria':<40} {'Registros':>15} {'%':>10}")
    print("-"*100)
    print(f"{'1. CAPÍTULO 44':<40} {len(df_cap_44):>15,} {(len(df_cap_44)/total_registros*100):>9.2f}%")
    print(f"{'2. NÃO CLASSIFICADO (OUTROS)':<40} {len(df_outros):>15,} {(len(df_outros)/total_registros*100):>9.2f}%")
    
    if len(df_eliminados) > 0:
        print(f"{'3. ELIMINADOS (Lixo/Outros Cap)':<40} {len(df_eliminados):>15,} {(len(df_eliminados)/total_registros*100):>9.2f}%")
    
    print("-"*100)
    print(f"{'TOTAL':<40} {total_registros:>15,} {100.0:>9.2f}%")
    
    # 3. ESTATÍSTICAS FINANCEIRAS
    print("\n" + "="*100)
    print("2. RESUMO GERAL - VALOR FINANCEIRO")
    print("="*100)
    
    total_valor = df_pandas['valor_total_vuntrib'].sum()
    valor_cap44 = df_cap_44['valor_total_vuntrib'].sum()
    valor_outros = df_outros['valor_total_vuntrib'].sum()
    valor_eliminados = df_eliminados['valor_total_vuntrib'].sum() if len(df_eliminados) > 0 else 0
    
    print(f"\n{'Categoria':<40} {'Valor Total (R$)':>20} {'%':>10}")
    print("-"*100)
    print(f"{'1. CAPÍTULO 44':<40} {valor_cap44:>20,.2f} {(valor_cap44/total_valor*100):>9.2f}%")
    print(f"{'2. NÃO CLASSIFICADO (OUTROS)':<40} {valor_outros:>20,.2f} {(valor_outros/total_valor*100):>9.2f}%")
    
    if len(df_eliminados) > 0:
        print(f"{'3. ELIMINADOS (Lixo/Outros Cap)':<40} {valor_eliminados:>20,.2f} {(valor_eliminados/total_valor*100):>9.2f}%")
    
    print("-"*100)
    print(f"{'TOTAL':<40} {total_valor:>20,.2f} {100.0:>9.2f}%")
    
    # 4. DETALHAMENTO DO CAPÍTULO 44 (por categoria)
    print("\n" + "="*100)
    print("3. DETALHAMENTO - CAPÍTULO 44 (Por Categoria)")
    print("="*100)
    
    if len(df_cap_44) > 0:
        # Análise por categoria
        analise_cap44 = df_cap_44.groupby('categoria_regex').agg({
            'prod_xprod': 'count',
            'valor_total_vuntrib': 'sum'
        }).rename(columns={
            'prod_xprod': 'Registros',
            'valor_total_vuntrib': 'Valor_Total'
        })
        
        analise_cap44['%_Registros'] = (analise_cap44['Registros'] / len(df_cap_44)) * 100
        analise_cap44['%_Valor'] = (analise_cap44['Valor_Total'] / valor_cap44) * 100
        
        # Ordenar por volume
        analise_cap44 = analise_cap44.sort_values('Registros', ascending=False)
        
        print(f"\n{'Categoria Cap 44':<35} {'Registros':>12} {'%Vol':>8} {'Valor (R$)':>18} {'%Fin':>8}")
        print("-"*100)
        
        for categoria, row in analise_cap44.iterrows():
            print(f"{categoria:<35} "
                  f"{row['Registros']:>12,.0f} "
                  f"{row['%_Registros']:>7.2f}% "
                  f"{row['Valor_Total']:>18,.2f} "
                  f"{row['%_Valor']:>7.2f}%")
        
        print("-"*100)
        print(f"{'TOTAL CAP 44':<35} "
              f"{len(df_cap_44):>12,} "
              f"{100.0:>7.2f}% "
              f"{valor_cap44:>18,.2f} "
              f"{100.0:>7.2f}%")
    else:
        print("\n⚠ Nenhum registro classificado como Capítulo 44")
    
    # 5. ANÁLISE DOS NÃO CLASSIFICADOS
    print("\n" + "="*100)
    print("4. ANÁLISE - NÃO CLASSIFICADOS (Amostra)")
    print("="*100)
    
    if len(df_outros) > 0:
        print(f"\nTotal não classificados: {len(df_outros):,}")
        print(f"Valor total: R$ {valor_outros:,.2f}")
        print(f"\nPrimeiros 20 exemplos:")
        print("-"*100)
        
        amostra_outros = df_outros.head(20)
        for idx, row in amostra_outros.iterrows():
            print(f"  {row['prod_xprod'][:80]}")
    else:
        print("\n✓ Todos os registros foram classificados!")
    
    # 6. SALVAR DATASETS SEPARADOS
    print("\n" + "="*100)
    print("5. SALVANDO DATASETS")
    print("="*100)
    
    df_cap_44.to_csv('dataset_cap44.csv', index=False, encoding='utf-8-sig')
    print(f"✓ Dataset Cap 44 salvo: dataset_cap44.csv ({len(df_cap_44):,} registros)")
    
    df_outros.to_csv('dataset_nao_classificados.csv', index=False, encoding='utf-8-sig')
    print(f"✓ Dataset Não Classificados salvo: dataset_nao_classificados.csv ({len(df_outros):,} registros)")
    
    if len(df_eliminados) > 0:
        df_eliminados.to_csv('dataset_eliminados.csv', index=False, encoding='utf-8-sig')
        print(f"✓ Dataset Eliminados salvo: dataset_eliminados.csv ({len(df_eliminados):,} registros)")
    
    # Salvar resumo
    resumo = {
        'categoria': ['Cap 44', 'Não Classificados', 'Eliminados', 'TOTAL'],
        'registros': [len(df_cap_44), len(df_outros), len(df_eliminados), total_registros],
        'pct_registros': [
            (len(df_cap_44)/total_registros)*100,
            (len(df_outros)/total_registros)*100,
            (len(df_eliminados)/total_registros)*100 if len(df_eliminados) > 0 else 0,
            100.0
        ],
        'valor_total': [valor_cap44, valor_outros, valor_eliminados, total_valor],
        'pct_valor': [
            (valor_cap44/total_valor)*100,
            (valor_outros/total_valor)*100,
            (valor_eliminados/total_valor)*100 if valor_eliminados > 0 else 0,
            100.0
        ]
    }
    
    df_resumo = pd.DataFrame(resumo)
    df_resumo.to_csv('resumo_analise_cap44.csv', index=False, encoding='utf-8-sig')
    print(f"✓ Resumo geral salvo: resumo_analise_cap44.csv")
    
    # 7. CONCLUSÕES
    print("\n" + "="*100)
    print("6. CONCLUSÕES")
    print("="*100)
    
    taxa_classificacao = (len(df_cap_44) / total_registros) * 100
    taxa_valor_cap44 = (valor_cap44 / total_valor) * 100
    
    print(f"""
    ✓ Taxa de Classificação Cap 44:
      - Volume: {taxa_classificacao:.2f}% dos registros
      - Valor: {taxa_valor_cap44:.2f}% do valor total
    
    ✓ Registros Não Classificados:
      - Volume: {(len(df_outros)/total_registros)*100:.2f}%
      - Valor: {(valor_outros/total_valor)*100:.2f}%
      
    ✓ Próximos Passos:
      1. Revisar registros não classificados em: dataset_nao_classificados.csv
      2. Analisar principais categorias Cap 44 identificadas
      3. Refinar regex se necessário para capturar mais itens
    """)
    
    print("="*100)
    print("ANÁLISE CONCLUÍDA!")
    print("="*100)
    
    return {
        'df_cap_44': df_cap_44,
        'df_outros': df_outros,
        'df_eliminados': df_eliminados,
        'resumo': df_resumo
    }

# ============================================================================
# VERSÃO SIMPLIFICADA (APENAS CONTAGEM)
# ============================================================================

def analise_simples(df_pandas):
    """Versão simplificada da análise - apenas contagens"""
    
    # Separar datasets
    df_cap_44 = df_pandas[df_pandas['categoria_regex'].isin(CATEGORIAS_CAP_44)].copy()
    df_outros = df_pandas[df_pandas['categoria_regex'] == 'NAO_CLASSIFICADO'].copy()
    df_eliminados = df_pandas[df_pandas['categoria_regex'].isin(CATEGORIAS_ELIMINADAS)].copy()
    
    # Imprimir resultado
    print("="*80)
    print("RESUMO DE CLASSIFICAÇÃO")
    print("="*80)
    print(f"1. CAPÍTULO 44:                  {len(df_cap_44):>10,}".replace(",", "."))
    print(f"2. NÃO CLASSIFICADO (OUTROS):    {len(df_outros):>10,}".replace(",", "."))
    
    if len(df_eliminados) > 0:
        print(f"3. ELIMINADOS (Lixo/Outros Cap): {len(df_eliminados):>10,}".replace(",", "."))
    
    print(f"{'─'*80}")
    print(f"TOTAL DE ITENS PROCESSADOS:      {len(df_pandas):>10,}".replace(",", "."))
    print("="*80)
    
    return df_cap_44, df_outros, df_eliminados

# ============================================================================
# EXEMPLO DE USO
# ============================================================================

if __name__ == "__main__":
    # Exemplo de uso
    
    # Opção 1: Análise Completa
    print("OPÇÃO 1: ANÁLISE COMPLETA")
    resultados = analisar_representatividade_completa(df_pandas)
    
    # Opção 2: Análise Simples (apenas contagens)
    print("\nOPÇÃO 2: ANÁLISE SIMPLES")
    print("="*80)
    # df_cap_44, df_outros, df_eliminados = analise_simples(df_pandas)
    
    '''print("""
    
    ╔════════════════════════════════════════════════════════════════════════╗
    ║                    INSTRUÇÕES DE USO                                   ║
    ╚════════════════════════════════════════════════════════════════════════╝
    
    1. Certifique-se de ter executado:
       df_pandas['categoria_regex'] = df_pandas['prod_xprod'].apply(categorizar_produto)
    
    2. Para análise completa:
       resultados = analisar_representatividade_completa(df_pandas)
    
    3. Para análise simples (apenas contagem):
       df_cap_44, df_outros, df_eliminados = analise_simples(df_pandas)
    
    4. Os resultados estarão em:
       - dataset_cap44.csv
       - dataset_nao_classificados.csv
       - resumo_analise_cap44.csv
    
    """)'''


In [ ]:
# Para validar
import pandas as pd
import re

# --- 1. DEFINIÇÃO DOS DICIONÁRIOS DE TERMOS (BASE PARA O REGEX) ---

# Exclusão absoluta (Serviços, Financeiro e Lixo de base)
termos_excl_prioritaria = 'ENTREGA|TAXA|FRETE|COMPLEMENTO DE VALOR|LOTE MISTO|MERCADORIA MISTA|SERVICO|VALOR ADICIONAL|AJUSTE'

# Materiais que invalidam o item (A menos que MADEIRA esteja presente)
termos_excl_materiais = 'VELUDO|PLASTICO|METAL|ALUMINIO|FERRO|ACO|VIDRO|BLINDEX|FUME|ACRILICO'

# Utensílios e Artefatos (Agora permitidos se validados por MADEIRA/BAMBU)
termos_artefatos_cap44 = 'CABIDE|ESPETO|ESPETINHO|CAKE BOARD|BANDEJA|CABO|FERRAMENTA|CAIXA|ADORNOS|PRENDEDOR|CAIXOTE|ESTRADO|PALITO|CEPO'

# Espécies Nobres e Validadas
termos_especies = 'CAMBARA|ANGELIM|TAUARI|CEDRINHO|CURUPIXA|ITAUBA|TATAJUBA|CUMARU|TECA|ABIURANA|CUPIUBA|AMESCLA|PINUS|EUCALIPTO|ARAUCO|GREENPLAC|CEDRO|MOGNO|VIROLA'

# Termos Técnicos e Processamento
termos_processamento = 'BENEFICIADA|SERRADA|SERRADO|VIGA|TABUA|TABUAS|TRONCO|FLORESTAL|CASCA|TORAS|TORA|CAVACO|LENHA|RESIDUO|BIOMASSA|MOINHA|18MM|1100X2200|FSC|CERTIFICADA|CERTIFICADO'

# Exclusões específicas de categorias
termos_excl_janela = 'INOX|PVC|APLIQUE|RECORTE|LASER|MINIATURA|BONECA|POLLY|BRINQUEDO|ARTESANATO|ENFEITE|DECORACAO|GELADEIRA|CORTINA|PERSIANA|TELA|MOSQUITEIRO'
termos_excl_carvao = 'ATIVADO|ATIVO|MINERAL|ANTRACIT|COQUE|HULHA|LINHITA|TURFA|FILTRO|MASCARA|SHAMPOO|SABONETE|CAPSULA|COMPRIMIDO'
termos_excl_mdf_moveis = 'ARMARIO|ESTANTE|GAVETEIRO|MESA|CADEIRA|BALCAO|CRIADO|GUARDA ROUPA|COZINHA|PLANEJADA|PLANEJADOS'

# --- 2. ESTRUTURA DAS CATEGORIAS REGEX ---

categorias_regex = [
    # A. BLOQUEIO DE SERVIÇOS E FINANCEIRO (Prioridade Máxima)
    ('OUTLIER__FINANCEIRO', rf'.*({termos_excl_prioritaria}).*'),
    
    # B. MÁQUINAS E VEÍCULOS
    ('OUTLIER__MAQUINAS', r'(PICADOR|TRATOR|COLHEITADEIRA|MÁQUINA|MAQUINA|VEÍCULO|VEICULO|PICADOR FLORESTAL)'),
    
    # C. ROTULADOS ANTES (Sua regra original)
    ('ROTULADOS ANTES', r'(LENHA|TORA DE MADEIRA|MADEIRA CERRADA|CABO DE MADEIRA|ARTIGO DOMESTICO DE MADEIRA|CARVAO VEGETAL|PORTA-BATENTE DE MADEIRA|PALITO DE DENTE|PALLET-EMBALAGEM DE MADEIRA|CHAPAS DE MDF|CHAPAS DE COMPENSADO|CHAPAS DE TAPUME|RESIDUO DE MADEIRA)'),

    # D. CAP 44 - ARTEFATOS, UTENSÍLIOS E CABOS (Regra de Material Concorrente)
    # Aceita se tiver o objeto E (Madeira/Bambu), mas REJEITA se tiver material concorrente (Veludo/Plástico) sem menção a madeira
    ('44.21.ARTEFATOS_MADEIRA', rf'^(?!.*({termos_excl_materiais})(?!.*(MADEIRA|MAD|MDF|BAMBU)))(?=.*({termos_artefatos_cap44}))(?=.*(MADEIRA|MAD|MDF|BAMBU|CERTIFICADA|FSC)).*'),

    # E. CAP 44 - PORTAS E JANELAS (Com exclusão de materiais e validação de espécies/medidas)
    ('44.18.PORTA_JANELA', rf'^(?!.*({termos_excl_materiais}|{termos_excl_janela})(?!.*(MADEIRA|MAD|MDF)))(?=.*(PORTA|JANELA|BATENTE|CAIXILHO|FORRO|SOALHO|PISO|ASSOALHO))(?=.*(MADEIRA|MAD|MDF|{termos_especies}|210|80X|70X|90X)).*'),

    # F. CAP 44 - PAINÉIS (MDF, MDP, COMPENSADO)
    # Regra: Compensado entra, mas se for PLASTIFICADO é excluído desta categoria
    ('44.11.MDF_PAINEL', rf'^(?!.*(PLASTIFICADO|{termos_excl_mdf_moveis}))(?=.*(MDF|MDP|OSB|HDF|COMPENSADO|ARAUCO|GREENPLAC))(?=.*(18MM|15MM|3MM|1100X2200|MADEIRA|MAD)).*'),

    # G. CAP 44 - CARVÃO VEGETAL
    ('44.02.CARVAO_VEGETAL', rf'^(?=.*(CARVÃO|CARVAO|MOINHA))(?!.*({termos_excl_carvao})).*'),

    # H. CAP 44 - MATÉRIA-PRIMA, SERRADOS E FLORESTAL (Inclusão Direta)
    ('44.03.MADEIRA_BRUTA_SERRADA', rf'.*({termos_especies}|{termos_processamento}).*'),

    # I. CAP 44 - PALETES E EMBALAGENS
    ('44.21.PALETE_PALLET', r'^(?=.*(PALETE|PALLET))(?!.*(LOCACAO|ALUGUEL|REFORMA|FRETE)).*'),

    # J. CATEGORIA DE REVISÃO (Itens ambíguos que precisam de olhar humano)
    ('AMBIGUOS_REVISAR', r'.*(PLANEJADA|COZINHA|DALMOBILE|LOTE MISTO|MERCADORIA MISTA).*'),

    ('OUTROS', r'.*')
]

# --- 3. FUNÇÃO DE PROCESSAMENTO ---

def categorizar_produto(descricao):
    descricao_upper = str(descricao).upper()
    for categoria, regex in categorias_regex:
        if re.search(regex, descricao_upper):
            return categoria
    return 'NAO_CLASSIFICADO'

# Aplicação no DataFrame
df_pandas['valor_total_vuntrib'] = df_pandas['prod_vuntrib'] * df_pandas['prod_qtrib'] * df_pandas['contagem']
df_pandas['categoria_regex'] = df_pandas['prod_xprod'].apply(categorizar_produto)

print("Processamento concluído com as novas regras do Capítulo 44.")

In [ ]:
categorias_cap_44 = [
    'ROTULADOS ANTES', '44.18.JANELA_MADEIRA', '44.02.CARVAO_VEGETAL', 
    '44.21.PALETE_PALLET', '44.11.MDF_PAINEL', 'NOMES_AMBÍGUOS', 
    '44.03.MADEIRA_BRUTA', '44.18.FOLHA_PORTA', '44.01.CAVACO_RESIDUO', 
    '44.21.OUTRAS_OBRAS'
]
categorias_eliminadas = ['OUTLIER__MAQUINAS', 'AMBIGUOS_EXCLUSAO']

df_cap_44 = df_pandas[df_pandas['categoria_regex'].isin(categorias_cap_44)].copy()
df_outros = df_pandas[df_pandas['categoria_regex'] == 'OUTROS'].copy()
df_eliminados = df_pandas[df_pandas['categoria_regex'].isin(categorias_eliminadas)].copy()

df_cap_44.to_csv('base_treino_nova_tratamento.csv', sep=';', index=False, encoding='utf-8-sig')
df_outros.to_csv('base_outros_nova_tratamento.csv', sep=';', index=False, encoding='utf-8-sig')
df_eliminados.to_csv('base_eliminados_lixo_nova_tratamento.csv', sep=';', index=False, encoding='utf-8-sig')

print(f"1. CAPÍTULO 44: {len(df_cap_44):>10,}".replace(",", "."))
print(f"2. OUTROS:        {len(df_outros):>10,}".replace(",", "."))
print(f"3. ELIMINADOS (Lixo/Outros Cap): {len(df_eliminados):>10,}".replace(",", "."))
print(f"TOTAL DE ITENS PROCESSADOS:      {len(df_pandas):>10,}".replace(",", "."))


In [ ]:
print(" Distribuição de Produtos por Categoria Regex (Top 20)")
distribuicao_regex = df_pandas['categoria_regex'].value_counts().head(20)
print(distribuicao_regex)

print(" Representatividade Financeira por Categoria (Top 20)")
representatividade_regex = (
    df_pandas.groupby('categoria_regex')['valor_total_vuntrib']
    .sum()
    .sort_values(ascending=False)
)

total = representatividade_regex.sum()
representatividade_percentual = (representatividade_regex / total) * 100
print(representatividade_percentual.head(20).round(2).astype(str) +"%")

In [ ]:
# 1. Representatividade por CONTAGEM (Volume de Dados)
contagem_absoluta = df_pandas['categoria_regex'].value_counts()
total_linhas = len(df_pandas)

representatividade_contagem_pct = (contagem_absoluta / total_linhas) * 100
print(representatividade_contagem_pct.head(20).round(2).astype(str) +"%")

# 2. Comparação Direta (Contagem vs Financeiro) 
comparativo = pd.DataFrame({
    'Volume (%)': representatividade_contagem_pct,
    'Financeiro (%)': representatividade_percentual
})
print(comparativo.head(20).round(2))


## 6. Para análises específicas

In [ ]:
df_pandas.head()

In [ ]:
'''ITEM ="MDF"
base ="cap_44.csv"
mask = df_pandas.apply(lambda col: col.astype(str).str.contains(ITEM, case=False, na=False))
df_filtrado = df_pandas[mask.any(axis=1)]
arquivo_saida = f"{ITEM}.csv"
df_filtrado.to_csv(arquivo_saida, index=False)'''

In [ ]:
df_outros = df_pandas[df_pandas['categoria_regex'] =="OUTROS"]
arquivo_saida ="OUTROS.csv"
df_outros.to_csv(arquivo_saida, index=False)

In [ ]:
# Criar um DataFrame apenas com as colunas necessárias para validação
df_validacao = df_pandas[['prod_xprod', 'categoria_regex']].copy()

# Salvar em CSV para conferência manual
# Usamos sep='\t'
df_validacao.to_csv('validacao_categorias_ncm44.csv', sep='\t', index=False, encoding='utf-8-sig')

print("Arquivo 'validacao_categorias_ncm44.csv' gerado com sucesso!")
# Mostrar uma amostra rápida de cada categoria
for cat in df_validacao['categoria_regex'].unique():
    print(f" Amostra da Categoria: {cat}")
    print(df_validacao[df_validacao['categoria_regex'] == cat]['prod_xprod'].head(3).tolist())


 
## 7. Novas Ferramentas de NLP (BoW, TF-IDF e Embeddings)

### 2. Bag of Words (BoW)
Representação de contagem de palavras para o dataset principal e filtros específicos.

In [ ]:
# 2.1 Bag of Words com Visualização
print("Processando Bag of Words...")
vectorizer_bow = CountVectorizer(max_features=1000, stop_words=['DE', 'DA', 'DO', 'COM', 'PARA', 'EM'])
bow_matrix = vectorizer_bow.fit_transform(df_pandas['prod_xprod_norm'])

# Criando um DataFrame para visualizar os Top 30 termos
sum_words = bow_matrix.sum(axis=0)
words_freq = [(word, sum_words[0, idx]) for word, idx in vectorizer_bow.vocabulary_.items()]
words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)

df_bow_viz = pd.DataFrame(words_freq, columns=['Termo', 'Frequência'])
print("\nTop 30 Termos - Bag of Words:")
display(df_bow_viz.head(30))

### 3. TF-IDF
Representação de frequência inversa de documentos para o dataset principal e filtros específicos.

In [ ]:
# 3.1 TF-IDF com Visualização
print("Processando TF-IDF...")

vectorizer_tfidf = TfidfVectorizer(
    max_features=1000, 
    max_df=0.7,  # IGNORE termos que aparecem em mais de 70% dos itens
    min_df=5,    # IGNORE termos que aparecem em menos de 5 itens
    ngram_range=(1, 2) # Captura"MADEIRA PINUS"como um único termo
)
tfidf_matrix = vectorizer_tfidf.fit_transform(df_pandas['prod_xprod_norm'])

# Criando um DataFrame para visualizar os Top 30 termos por Score Médio
feature_names = vectorizer_tfidf.get_feature_names_out()
avg_tfidf = tfidf_matrix.mean(axis=0).tolist()[0]
tfidf_scores = sorted(list(zip(feature_names, avg_tfidf)), key=lambda x: x[1], reverse=True)

df_tfidf_viz = pd.DataFrame(tfidf_scores, columns=['Termo', 'Score_TFIDF'])
print("\nTop 30 Termos - TF-IDF (Importância Relativa):")
display(df_tfidf_viz.head(30))

### 4. Word Embeddings
Implementação de Word2Vec e FastText.

In [ ]:
def simple_tokenize(text):
    return re.findall(r'\w+', str(text).upper())

sentences = df_pandas['prod_xprod_norm'].apply(simple_tokenize).tolist()
print(f"Tokenização: {len(sentences)} sentenças preparadas.")

# 4.1 Word2Vec
print("Treinando Word2Vec")
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=5, workers=4)

termo_teste = 'MADEIRA'
if termo_teste in w2v_model.wv:
    print(f"\nPalavras mais similares a '{termo_teste}':")
    display(pd.DataFrame(w2v_model.wv.most_similar(termo_teste, topn=20), columns=['Termo', 'Similaridade']))

In [ ]:
# 4.2 FastText - Treinamento
print("Treinando FastText (lidando com sub-palavras e erros de digitação)...")
ft_model = FastText(sentences, vector_size=100, window=5, min_count=5, workers=4)
print("FastText treinado com sucesso!")

termo_teste_ft = 'MADEIRA'
if termo_teste_ft in ft_model.wv:
    print(f"\nPalavras mais similares a '{termo_teste_ft}' (via FastText):")
    display(pd.DataFrame(ft_model.wv.most_similar(termo_teste_ft, topn=30), columns=['Termo', 'Similaridade']))

In [ ]:
# 4.2 FastText - Treinamento
print("Treinando FastText (lidando com sub-palavras e erros de digitação)...")
ft_model = FastText(sentences, vector_size=100, window=5, min_count=5, workers=4)
print("FastText treinado com sucesso!")

termo_teste_ft = 'JANELA'
if termo_teste_ft in ft_model.wv:
    print(f"\nPalavras mais similares a '{termo_teste_ft}' (via FastText):")
    display(pd.DataFrame(ft_model.wv.most_similar(termo_teste_ft, topn=30), columns=['Termo', 'Similaridade']))

## Exportação de dados financeiros para análise

In [ ]:
df = df_pandas

# 2. NORMALIZAÇÃO E LIMPEZA
df['prod_xprod_norm'] = df['prod_xprod'].astype(str).str.upper()
df['prod_vuntrib'] = pd.to_numeric(df['prod_vuntrib'], errors='coerce').fillna(0)
df['prod_qtrib'] = pd.to_numeric(df['prod_qtrib'], errors='coerce').fillna(0)
df['contagem'] = pd.to_numeric(df['contagem'], errors='coerce').fillna(1)

# Calcular valor total por linha (Valor Unitário * Quantidade de Ocorrências)
df['valor_total_vuntrib'] = df['prod_vuntrib'] * df['contagem']
df['valor_total_qtrib'] = df['prod_qtrib'] * df['contagem']

# Stopwords a ignorar (Palavras irrelevantes para a análise financeira)
stopwords = {
    'DE', 'DA', 'DO', 'COM', 'PARA', 'EM', 'UM', 'UMA', 'CM', 'MM', 'MT', 
    'KG', 'UN', 'UND', 'C/', 'X', 'O', 'A', 'OS', 'AS', 'PARA', 'POR', 'NAS'
}

# Dicionário para acumular estatísticas: {palavra: [soma_vuntrib, soma_qtrib, contagem_ocorrencias]}
word_stats = defaultdict(lambda: [0.0, 0.0, 0])

# 3. PROCESSAMENTO POR LINHA
for idx, row in df.iterrows():
    # Tokenização: extrai apenas palavras (letras e números)
    words = set(re.findall(r'\w+', row['prod_xprod_norm']))
    v_total = row['valor_total_vuntrib']
    q_total = row['valor_total_qtrib']
    
    for word in words:
        # Filtra stopwords, números puros e palavras muito curtas
        if word not in stopwords and not word.isdigit() and len(word) > 1:
            stats = word_stats[word]
            stats[0] += v_total
            stats[1] += q_total
            stats[2] += 1

# 4. CONSOLIDAÇÃO DOS RESULTADOS
result_data = []
for word, stats in word_stats.items():
    result_data.append({
        'Termo': word,
        'Valor_Total_Vuntrib': stats[0],
        'Valor_Total_Qtrib': stats[1],
        'Ocorrencias': stats[2],
        'Valor_Medio_Vuntrib': stats[0] / stats[2] if stats[2] > 0 else 0,
        'Valor_Medio_Qtrib': stats[1] / stats[2] if stats[2] > 0 else 0
    })

df_results = pd.DataFrame(result_data)

# 5. GERAÇÃO DOS TOP 50
# Nota: Para o valor médio, filtramos termos com menos de 10 ocorrências para evitar distorções por erros pontuais
top_50_total_vuntrib = df_results.sort_values(by='Valor_Total_Vuntrib', ascending=False).head(50)
top_50_medio_vuntrib = df_results[df_results['Ocorrencias'] > 10].sort_values(by='Valor_Medio_Vuntrib', ascending=False).head(50)
top_50_total_qtrib = df_results.sort_values(by='Valor_Total_Qtrib', ascending=False).head(50)
top_50_medio_qtrib = df_results[df_results['Ocorrencias'] > 10].sort_values(by='Valor_Medio_Qtrib', ascending=False).head(50)

# 6. SALVAMENTO DOS ARQUIVOS
top_50_total_vuntrib.to_csv('top_50_total_vuntrib_nova.csv', index=False)
top_50_medio_vuntrib.to_csv('top_50_medio_vuntrib_nova.csv', index=False)
top_50_total_qtrib.to_csv('top_50_total_qtrib_nova.csv', index=False)
top_50_medio_qtrib.to_csv('top_50_medio_qtrib_nova.csv', index=False)

print("Os 4 arquivos CSV com o Top 50 foram gerados.")


In [ ]:
def analisar_valor_transacao_por_termo(df):
    
    df['valor_transacao_total'] = df['prod_vuntrib'] * df['prod_qtrib'] * df['contagem']

    # Stopwords a ignorar
    stopwords = {
        'DE', 'DA', 'DO', 'COM', 'PARA', 'EM', 'UM', 'UMA', 'CM', 'MM', 'MT', 
        'KG', 'UN', 'UND', 'C/', 'X', 'O', 'A', 'OS', 'AS', 'POR', 'NAS', 'NOS',
        'E', 'OU', 'SE', 'NA', 'NO', 'NOS', 'NAS', 'AO', 'AOS', 'AS', 'ESTE', 'ESTA',
        'ESTES', 'ESTAS', 'ESSE', 'ESSA', 'ESSES', 'ESSAS', 'AQUELA', 'AQUELE', 'AQUELAS', 'AQUELOS'
    }

    word_stats = defaultdict(lambda: [0.0, 0])
    
    def processar_linha(row):
        words = set(re.findall(r'\w+', row['prod_xprod_norm']))
        v_transacao = row['valor_transacao_total']
        
        for word in words:
            # Filtros: Stopwords, dígitos e tamanho > 1
            if word not in stopwords and not word.isdigit() and len(word) > 1:
                stats = word_stats[word]
                stats[0] += v_transacao
                stats[1] += 1

    # Aplica a função de processamento linha a linha
    df.apply(processar_linha, axis=1)

    # 2. Consolidação
    result_data = []
    for word, stats in word_stats.items():
        result_data.append({
            'Termo': word,
            'Valor_Transacao_Total': stats[0],
            'Ocorrencias': stats[1],
            'Valor_Transacao_Medio': stats[0] / stats[1] if stats[1] > 0 else 0
        })

    df_results = pd.DataFrame(result_data)

    # 3. Geração dos Top 100
    
    # Top 100 Total
    top_100_total = df_results.sort_values(by='Valor_Transacao_Total', ascending=False).head(100)
    top_100_total.to_csv('top_100_total_transacao_nova.csv', index=False, encoding='utf-8-sig')
    
    # Top 100 Médio
    top_100_medio = df_results[df_results['Ocorrencias'] >= 10].sort_values(by='Valor_Transacao_Medio', ascending=False).head(100)
    top_100_medio.to_csv('top_100_medio_transacao_nova.csv', index=False, encoding='utf-8-sig')

    print(f"Total de termos únicos analisados: {len(df_results)}")
    
    return df_results


In [ ]:
if 'df_pandas' in locals() and 'prod_xprod_norm' in df_pandas.columns:
    df_termos_analisados = analisar_valor_transacao_por_termo(df_pandas)
else:
    print("Erro: O DataFrame 'df_pandas' ou a coluna 'prod_xprod_norm' não foram encontrados. Certifique-se de executar as células anteriores.")